In [1]:
# Task 3.1 (4m)
#1m - Create question table with PK
#1m - Create scores table with PK, where score field is integer
import sqlite3

conn = sqlite3.connect('quiz.db')

conn.execute("DROP TABLE IF EXISTS question")
conn.execute("DROP TABLE IF EXISTS scores")

conn.execute("""
CREATE TABLE question (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    question_text TEXT,
    option_a TEXT,
    option_b TEXT,
    option_c TEXT,
    option_d TEXT,
    correct_option TEXT
)
""")

conn.execute("""
CREATE TABLE scores (
    username TEXT PRIMARY KEY,
    score INTEGER
)
""")
conn.commit()

#1m - process question.csv
#1m - ... and insert questions into the question table

file = open("questions.csv")
next(file)
for line in file:
    conn.execute("""INSERT INTO question (question_text, option_a, option_b, option_c, option_d, correct_option)
                VALUES(?,?,?,?,?,?)""", line.strip().split(","))


file.close()
conn.commit()
conn.close()


In [2]:
#Task 3.2 - (3m)

class Question: 
    #1m - Constructor
    def __init__(self, question, optiona, optionb, optionc, optiond, correctopt):
        self.question = question
        self.optiona = optiona
        self.optionb = optionb
        self.optionc = optionc
        self.optiond = optiond
        self.correctopt = correctopt

    def to_str(self): # 1m - to_str() method that returns formatted string of the question and options	
        qstr = self.question + "\n"
        
        qstr += "A. " + self.optiona + "\n"
        qstr += "B. " + self.optionb + "\n"
        qstr += "C. " + self.optionc + "\n"
        qstr += "D. " + self.optiond + "\n"
        qstr += "Your answer (A/B/C/D): "
        return qstr
    
    def check_answer(self, userchoice): # 1m - check_answer() method that returns T/F	 
        choice = userchoice.upper()
        return choice in ['A', 'B', 'C', 'D'] and choice == self.correctopt

    def get_answer(self): #optional - used to preserve encapsulation
        return self.correctopt

In [3]:
#Task 3.3 (4m)
#1m - Fetch questions from DB
#1m - Randomly select 5 questions
#1m - ... and return list of 5 `Questions` objects 
#1m - Show each `Question` object using to_str() method
import random
import sqlite3
def generate_5_questions():
    conn = sqlite3.connect("quiz.db")
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM questions")
    rows = cursor.fetchall()
    conn.close()

    selected = random.sample(rows, 5)
    question_lst = []
    for row in selected:
        q = Question(row[1],
                     row[2],
                     row[3],
                     row[4],
                     row[5],
                     row[6])
        question_lst.append(q)
    return question_lst


qns = generate_5_questions()
for q in qns:
    print(q.to_str())
        

What is 2 + 2?
A. 3
B. 4
C. 5
D. 6
Your answer (A/B/C/D): 
What color is the sky on a clear day?
A. Red
B. Blue
C. Green
D. Yellow
Your answer (A/B/C/D): 
What is the chemical symbol for water?
A. H2O
B. O2
C. CO2
D. NaCl
Your answer (A/B/C/D): 
What is the largest planet?
A. Earth
B. Mars
C. Jupiter
D. Saturn
Your answer (A/B/C/D): 
What is the capital of France?
A. Berlin
B. Madrid
C. Paris
D. Rome
Your answer (A/B/C/D): 


In [4]:
# Task 3.4 - (5m)
#1m - QuizSession class with appropriate constructor and default value
#1m - if user has previous attempt, 
#1m - ... if current score outperforms previous score, update DB and return appropriate message
#1m - ... else current score is less than / equal, return appropriate message
#1m - else, insert new score for new user

import sqlite3
class QuizSession:
    def __init__(self, username):
        self.username = username
        self.score = 0

    def updateScoresDB(self):
        message = ""
        dbconn = sqlite3.connect("quiz.db")
        cursor = dbconn.cursor()
        cursor.execute("SELECT score FROM scores WHERE username = ?", (self.username,))
        row = cursor.fetchone()
        if row:
            if self.score > row[0]:
                cursor.execute("UPDATE scores SET score = ? WHERE username = ?", (self.score, self.username))
                message += "Congratulations, you have acheived a new high score!\n"
            else:
                message += "You did not outperform your previous score.\n"
        else:
            dbconn.execute("INSERT INTO scores (username, score) VALUES(?, ?)", (self.username, self.score))
            message += "Your score has been updated on quiz.db\n"
        dbconn.commit()
        dbconn.close()
        return message

    def incScore(self): #optional - provided to ensure encapsulation
        self.score += 1
# Test
test = QuizSession("James")
test.score = 3
print(test.updateScoresDB())
test.score = 2
print(test.updateScoresDB())
test.score = 5
print(test.updateScoresDB())

Your score has been updated on quiz.db

You did not outperform your previous score.

Congratulations, you have acheived a new high score!



In [5]:
#Task 3.5 (4m)
# 1m - Order BY score Desc
# 1m - ... username ASC
# 1m - LIMIT 3
# 1m - Formatted output returned
def getLeaderBoard():
    dbconn = sqlite3.connect("quiz.db")
    cursor = dbconn.cursor()
    cursor.execute("SELECT username, score FROM scores ORDER BY score DESC, username ASC LIMIT 3")
    rows = cursor.fetchall()
    dbconn.close()
    leaderboard_str = f"Top 3 players\n{'='*15}\n"
    for r in rows:
        leaderboard_str += f"{r[0]}: {r[1]}pts\n"
    return leaderboard_str
        

In [ ]:
#Task 3.6 (10m)
# 1m - Accept a new client connection
# 1m - Request for username from the client
# 1m - Create a new QuizSession object for the username
# 1m - Sends 5 questions (1 at a time) to the client and receives answers.
# 1m - Validates answers and updates score in the QuizSession object
# 1m - Report final score to client
# 1m - Updates the scores table in quiz.db using updateScoreDB() method 
# 1m - Sends client the leaderboard information, using getLeaderboard()function.
# 1m - Close connection with client
# 1m - Show output of client/server interaction

import socket

server_socket= socket.socket()
server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server_socket.bind(('127.0.0.1', 12345)) 
server_socket.listen()
print(f"Quiz Server running on 127.0.0.1:12345")

try:
    while True:
        conn, addr = server_socket.accept()
        print(f"Connected by {addr}")
        conn.sendall(b"Enter your name: ") #1m - Ask for user name
        username = conn.recv(1024).decode().strip()
        print(f"{username} connected from {conn}")
        session = QuizSession(username)    # ... and setup new QuizSession
        questions = generate_5_questions() #1m - generate q
    
        for q in questions:
            conn.sendall(q.to_str().encode())
            answer = conn.recv(1024).decode().strip() #1m
            if q.check_answer(answer):
                session.incScore()
                conn.sendall(b"Correct!\n")
            else:
                conn.sendall(f"Wrong! Correct answer: {q.get_answer()}\n".encode())
    
        conn.sendall(f"\nYour total score: {session.score}/5\n".encode())    
        message = session.updateScoresDB()
        conn.sendall(message.encode())
        conn.sendall(getLeaderBoard().encode())
        conn.sendall("\nThank you for participating!\nConnection is closed.".encode())
        conn.close()
except KeyboardInterrupt:
        print("\nShutdown server.")    

        
    

Quiz Server running on 127.0.0.1:12345
Connected by ('127.0.0.1', 59923)
Julie connected from <socket.socket fd=80, family=2, type=1, proto=0, laddr=('127.0.0.1', 12345), raddr=('127.0.0.1', 59923)>
